# 02 Prepare Detection Dataset

Purpose: create reproducible train/validation/test splits and export detection datasets for YOLO and COCO.

**Inputs**

- Cleaned bounding-box dataset at `FISHDETECT_DATASET_ROOT`
- `configs/experiments.yaml`

**Outputs**

- `FISHDETECT_PREPARED_ROOT/metadata/`
- `FISHDETECT_PREPARED_ROOT/yolo_det/`
- `FISHDETECT_PREPARED_ROOT/coco_det/`

No segmentation or mask export is part of the active workflow.


In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "scripts").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Could not find repo root. Open the notebook from inside the fishDetect repository.")

REPO = find_repo_root()
os.chdir(REPO)

DATASET_ROOT = Path(os.environ.get(
    "FISHDETECT_DATASET_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/merged_viame_v2"
))

PREPARED_ROOT = Path(os.environ.get(
    "FISHDETECT_PREPARED_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/prepared_ml"
))

os.environ["FISHDETECT_DATASET_ROOT"] = str(DATASET_ROOT)
os.environ["FISHDETECT_PREPARED_ROOT"] = str(PREPARED_ROOT)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

print("Repo root:", REPO)
print("Dataset root:", DATASET_ROOT)
print("Prepared root:", PREPARED_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())
print("Prepared parent exists:", PREPARED_ROOT.parent.exists())


## Run Preparation

This command reads the master dataset and writes prepared outputs to `FISHDETECT_PREPARED_ROOT`. It does not modify the master dataset.


In [ ]:
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset root not found: {DATASET_ROOT}. Set FISHDETECT_DATASET_ROOT and rerun.")

!python scripts/prepare_dataset.py --config configs/experiments.yaml


## Verify Prepared Outputs


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

expected = [
    PREPARED_ROOT / "metadata" / "annotations.csv",
    PREPARED_ROOT / "metadata" / "images.csv",
    PREPARED_ROOT / "metadata" / "split_manifest.csv",
    PREPARED_ROOT / "annotations_common.csv",
    PREPARED_ROOT / "yolo_det" / "data.yaml",
    PREPARED_ROOT / "coco_det" / "train.json",
    PREPARED_ROOT / "coco_det" / "val.json",
    PREPARED_ROOT / "coco_det" / "test.json",
]

for path in expected:
    print(f"{'OK' if path.exists() else 'MISSING':8} {path}")

summary_path = PREPARED_ROOT / "metadata" / "prepare_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("
Preparation summary")
    for key in ["image_count_exported", "annotation_count_exported", "class_count", "split_manifest", "yolo_data_yaml", "coco_root"]:
        print(f"{key}: {summary.get(key)}")


## Split and Class Balance


In [ ]:
split_summary_path = PREPARED_ROOT / "metadata" / "split_summary.csv"
class_split_path = PREPARED_ROOT / "metadata" / "class_counts_by_split.csv"
split_path = PREPARED_ROOT / "metadata" / "split_manifest.csv"

if split_summary_path.exists():
    display(pd.read_csv(split_summary_path))
else:
    print("Missing split summary. Run preparation first.")

if class_split_path.exists():
    counts = pd.read_csv(class_split_path)
    display(counts.pivot_table(index="class_name", columns="split", values="annotation_count", fill_value=0, aggfunc="sum").head(30))
else:
    print("Missing class counts by split. Run preparation first.")

if split_path.exists():
    split = pd.read_csv(split_path)
    leak_cols = [c for c in ["sha256_if_available", "source_file_or_dataset_if_available"] if c in split.columns]
    for col in leak_cols:
        valid = split[col].dropna().astype(str)
        valid = valid[valid != ""]
        if valid.empty:
            print(f"No values available for leakage check column: {col}")
            continue
        split_counts = split.loc[split[col].astype(str).isin(valid), [col, "split"]].drop_duplicates().groupby(col)["split"].nunique()
        leaks = split_counts[split_counts > 1]
        print(f"Leakage check by {col}: {'OK' if leaks.empty else 'POTENTIAL LEAKS'} ({len(leaks)} flagged)")
